# Eliminar un usuario

## Qué hace

Elimina el usuario de demostración.

## Qué pasa en la base de datos

Borra únicamente el registro identificado por el correo de demo.

## Qué acción usa

`deleteUser(user.id)`.

## Anotaciones

Ejecutar la celda elimina el usuario de demo. Si ya se eliminó, muestra User not found.

Selecciona la celda de abajo y pulsa **▶** o **Shift+Enter**. No necesitas
escribir código. El resultado aparece debajo en una tabla con encabezados cian.
Los datos de demostración ya están preparados en el documento.


In [ ]:
import json
import subprocess
from pathlib import Path

app = Path.cwd().resolve()
while app != app.parent and not (app / "logic").is_dir():
    app = app.parent
if not (app / "logic").is_dir():
    raise SystemExit("No encuentro la app. Abre Jupyter desde first-app.")

code = r"""
import { deleteUser } from "./logic/delete-user.js";
import { findUserByEmail } from "./logic/find-user-by-email.js";

try {
  const user = findUserByEmail("demo.jimena.jupyter@example.com");
  const result = deleteUser(user.id);
  console.log(JSON.stringify(result === undefined ? null : result));
} catch (error) {
  console.error(error.message);
  process.exitCode = 1;
}
"""

try:
    completed = subprocess.run(
        ["node", "--input-type=module", "--eval", code],
        cwd=app,
        capture_output=True,
        encoding="utf-8",
    )
except OSError:
    raise SystemExit("No encuentro Node.js. Revisa la preparación de la app.")

if completed.returncode != 0:
    print(completed.stderr.strip())
else:
    result = json.loads(completed.stdout)
    headers = ['Resultado']
    rows = [["Usuario eliminado. La acción no devuelve datos."]]
    # Tabla de texto con encabezados cian, sin librerías adicionales.
    rows = [[str(value).replace("\n", " ").replace("\r", " ").replace("\x1b", "") for value in row] for row in rows]
    widths = [max(len(header), *(len(row[index]) for row in rows)) if rows else len(header) for index, header in enumerate(headers)]
    print("┌" + "┬".join("─" * (width + 2) for width in widths) + "┐")
    print("│ " + " │ ".join("\x1b[1;36m" + header.ljust(width) + "\x1b[0m" for header, width in zip(headers, widths)) + " │")
    print("├" + "┼".join("─" * (width + 2) for width in widths) + "┤")
    for row in rows:
        print("│ " + " │ ".join(value.ljust(width) for value, width in zip(row, widths)) + " │")
    print("└" + "┴".join("─" * (width + 2) for width in widths) + "┘")
    if not rows:
        print("No hay filas.")
